In [ ]:
!git clone https://github.com/676627/deep-learning-eurosat.git
%cd deep-learning-eurosat
!pip install -q rasterio wandb

In [ ]:
# Download and unzip data
!wget -q "https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip"
!wget -q "https://zenodo.org/records/7711810/files/EuroSAT_MS.zip"

!unzip -q EuroSAT_RGB.zip -d data/
!unzip -q EuroSAT_MS.zip -d data/

In [ ]:
# Config - rgb or multispectral
MODE         = "ms"
MS_DATA_DIR  = "data/EuroSAT_MS"
# MODE = "rgb"
# RGB_DATA_DIR = "data/EuroSAT_RGB"
BAND_INDICES = None   # all 13 bands
EPOCHS       = 50
BATCH_SIZE   = 64
LR           = 1e-3




In [ ]:
# Cell — data
import sys
sys.path.insert(0, '..')
from src.dataset import load_ms_dataset
train_ds, val_ds, stats = load_ms_dataset(MS_DATA_DIR, batch_size=BATCH_SIZE)
n_channels = 13
run_name   = "MS_all13bands"

# from src.dataset import load_rgb_dataset
# train_ds, val_ds = load_rgb_dataset(RGB_DATA_DIR, batch_size=BATCH_SIZE)
# n_channels = 3
# run_name   = "RGB_baseline"

In [ ]:
# model + train
from src.model import build_model
import keras, wandb
from wandb.integration.keras import WandbMetricsLogger

model = build_model(in_channels=n_channels)
model.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
wandb.init(project="eurosat-cnn", name=run_name)
model.fit(
    train_ds, epochs=EPOCHS, validation_data=val_ds,
    callbacks=[
        WandbMetricsLogger(),
        keras.callbacks.ModelCheckpoint(
            f"{run_name}_best.keras",
            monitor="val_accuracy", save_best_only=True,
        ),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
    ],
)
wandb.finish()